#Sistema de Apoio à Decisão (SAD): Otimização de Matrícula Acadêmica
Projeto: Trilha EPR — Engenharia de Produção (UnB)

O Problema:
O planejamento semestral de disciplinas exige a conciliação de três variáveis complexas que impactam diretamente a retenção e o sucesso do aluno:

Identidade Profissional: Aderência das ementas aos objetivos de carreira do aluno, baseados nas áreas de conhecimento da ABEPRO.

Restrições Rígidas (Pré-requisitos): O cumprimento de uma sequência lógica de aprendizado (Cálculos, Físicas e matérias base).

Capacidade de Carga: O limite de horas disponíveis, crucial para alunos do período noturno ou em estágio.

A Solução: O Motor de Otimização Semântica
Diferente de sistemas de busca comuns, este SAD utiliza Processamento de Linguagem Natural (NLP) e Otimização Combinatória para sugerir o "Semestre Ideal". O motor não busca palavras-chave, ele interpreta a intenção de carreira do aluno e valida, em milissegundos, se ele possui os requisitos legais para cursar aquela sugestão.

In [2]:
!pip install sentence-transformers pandas ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.7 MB/s eta 0:00:00


#Etapa 1: ETL e Resiliência de Dados
Nesta etapa, o sistema realiza o processamento dos dados brutos (CSV). Devido às variações de base (como as geradas por ferramentas como Antigravity), implementamos um pipeline de dados resiliente:

Mapeamento Dinâmico: O código identifica colunas essenciais (Ementa, Código, Créditos) mesmo que os títulos mudem.

Deduplicação de Índices: Limpeza automática de códigos repetidos para garantir a integridade do dicionário de busca.

Normalização: Padronização de strings e tratamento de valores nulos para evitar falhas durante o cálculo matemático.


#Etapa 2: Inteligência Artificial e Similaridade de Cossenos
O coração do SAD é o modelo paraphrase-multilingual-MiniLM-L12-v2. A lógica de decisão segue os passos:

Vetorização (Embeddings): Transformamos textos de ementas e objetivos de carreira em vetores numéricos de alta dimensão.

Similaridade de Cossenos: Calculamos a proximidade angular entre o desejo do aluno e o conteúdo da disciplina.

Vantagem: Superamos o "vício de palavras" (overfitting lexical). Se o aluno quer "Finanças", o sistema recomenda "Engenharia Econômica" por proximidade semântica, mesmo sem a palavra exata.

Score de Aderência: Atribuímos uma nota de 0 a 100% para cada matéria disponível.


In [3]:
import pandas as pd
import io
import re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files
from sentence_transformers import SentenceTransformer, util

print("Carregando o Cérebro da IA (Sentence Transformers)...")
modelo_ia = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ==============================================================================
# 1. UPLOAD E HIGIENIZAÇÃO DA BASE (ETL)
# ==============================================================================
print("\nFaça o upload da sua base de dados original com ementas (CSV)...")
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]), sep=None, engine='python')

# Padronização e Mapeamento de Colunas
colunas_map = {col: col.strip().upper() for col in df.columns}
df = df.rename(columns=colunas_map)

col_codigo = next((c for c in df.columns if 'COD' in c or 'ID' in c), 'CODIGO')
col_nome = next((c for c in df.columns if 'NOME' in c), 'NOME')
col_req = next((c for c in df.columns if 'REQ' in c and 'CO' not in c), 'PRE_REQUISITOS')
col_cred = next((c for c in df.columns if 'CRED' in c or 'HORA' in c), 'CREDITO')
col_ementa = next((c for c in df.columns if 'EMENTA' in c or 'CONTEUDO' in c or 'DESCRICAO' in c), None)
if col_ementa is None:
    col_ementa = col_nome
col_tipo = next((c for c in df.columns if 'TIPO' in c), 'TIPO')

df[col_codigo] = df[col_codigo].astype(str).str.strip().str.upper()
df = df[df[col_codigo].notna() & (df[col_codigo] != 'NAN')]

#  SOLUÇÃO DE ERRO: Remove disciplinas duplicadas mantendo a primeira ocorrência
df = df.drop_duplicates(subset=[col_codigo], keep='first')

for col in [col_req, col_ementa, col_tipo]:
    if col in df.columns:
        df[col] = df[col].fillna('')

if col_cred in df.columns:
    df[col_cred] = pd.to_numeric(df[col_cred], errors='coerce').fillna(60).astype(int)

# Vetorização Semântica
print("\nVetorizando ementas para o cálculo de cossenos...")
def gerar_vetor(row):
    # Se a ementa for muito curta ou não existir, ignora
    texto = str(row.get(col_ementa, ''))
    if len(texto) > 3:
        return modelo_ia.encode(texto)
    return None

df['Vetor_Ementa'] = df.apply(gerar_vetor, axis=1)

# Salvando a base limpa para o Motor (S duplicatas)
dict_disciplinas = df.set_index(col_codigo).to_dict('index')

# Mini Raio-X Automático
vetores_validos = sum(1 for d in dict_disciplinas.values() if d.get('Vetor_Ementa') is not None)
print(f"Base carregada! Total de disciplinas únicas: {len(dict_disciplinas)}")
print(f"Total com ementas válidas para a IA: {vetores_validos}")

# ==============================================================================
# 2. MOTOR LÓGICO DE OTIMIZAÇÃO E PRÉ-REQUISITOS (O Trator Regex)
# ==============================================================================
def checar_requisitos(expressao, historico):
    if pd.isna(expressao) or str(expressao).strip() == '': return True
    req_limpo = re.sub(r'[()]', '', str(expressao)).strip().upper()
    req_limpo = re.sub(r'\s+E\s+', '+', req_limpo)
    req_limpo = re.sub(r'\s*,\s*', '+', req_limpo)
    req_limpo = re.sub(r'\s+OU\s+', '|', req_limpo)

    opcoes_ou = req_limpo.split('|')
    for opcao in opcoes_ou:
        combo_e = opcao.split('+')
        if all(sub.strip() in historico for sub in combo_e if sub.strip()):
            return True
    return False

descricoes_areas = {
    "Eng. de Operações e Processos": "Sistemas de produção, manufatura, layout e PCP.",
    "Logística": "Cadeia de suprimentos, transporte, modais e estoque.",
    "Pesquisa Operacional": "Otimização combinatória, simulação, modelos matemáticos.",
    "Eng. da Qualidade": "Gestão da qualidade, controle estatístico, seis sigma.",
    "Eng. Organizacional": "Gestão estratégica, projetos, empreendedorismo e RH.",
    "Tecnologia e Automação": "Programação, banco de dados, circuitos e automação."
}

Carregando o Cérebro da IA (Sentence Transformers)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Faça o upload da sua base de dados original com ementas (CSV)...


Saving disciplinas.csv to disciplinas.csv

Vetorizando ementas para o cálculo de cossenos...
Base carregada! Total de disciplinas únicas: 223
Total com ementas válidas para a IA: 223



#Etapa 3: O "Trator Regex" e Validação de Restrições
Para garantir que o suporte à decisão seja realista, implementamos um parser de lógica booleana via Expressões Regulares (Regex):

Lógica Booleana: O motor interpreta estruturas complexas como (MAT0025 E MAT0026) OU EST0023.

Filtro de Fluxo: Disciplinas que não possuem os requisitos liberados no histórico são redirecionadas ou bloqueadas.

Modo Exploratório vs. Rígido: O sistema permite "relaxar" as restrições para que o aluno visualize o seu futuro (Gap Analysis), identificando quais matérias precisa vencer hoje para liberar os sonhos de amanhã.

Etapa 4: Otimização da Mochila (Knapsack Problem)
A sugestão final é um problema clássico de otimização. O sistema tenta preencher o "espaço livre" (Carga Máxima - Carga Obrigatória) com as optativas de maior valor (Score de IA).

Resultado: Uma grade que maximiza o ganho profissional respeitando o limite humano de tempo do estudante.

In [4]:
# ==============================================================================
# 3. PAINEL DE CONFIGURAÇÃO DO MVP (Sem dependência de Widgets)
# ==============================================================================
# Altere as variáveis abaixo para simular os cenários acadêmicos:
#CENÁRIOS:
#Passou do segundo semestre: MAT0025, MAT0026, IFD0171, IFD0173, MAT0031, CIC0007
#Passou do quarto semestre: MAT0025, MAT0026, MAT0027, IFD0171, IFD0173, IFD0175, MAT0031, CIC0007, EST0023, ENC0053
#Passou do sexto semestre: MAT0025, MAT0026, MAT0027, IFD0171, IFD0173, IFD0175, MAT0031, CIC0007, EST0023, ENC0035, ADM0023, EPR0066
#Passou do oitavo semestre: MAT0025, MAT0026, MAT0027, IFD0171, IFD0173, IFD0175, MAT0031, CIC0007, EST0023, ENC0035, ADM0023, EPR0066, ENM0080, IGD0043, ADM0010


# 1. Histórico do Aluno (Códigos separados por vírgula, tambem há outros exemplos de histórico abaixo)
HISTORICO = "MAT0025, MAT0026, EST0023, IFD0171"

# 2. Trilha de Carreira Desejada
# Opções: "Eng. de Operações e Processos", "Logística", "Pesquisa Operacional",
# "Eng. da Qualidade", "Eng. Organizacional", "Tecnologia e Automação"
TRILHA_DESEJADA = "Logística"

# 3. Limites de Tempo e IA
CARGA_MAXIMA = 240       # Carga Máxima Suportada no Semestre (Horas)
CARGA_OBRIGATORIAS = 120 # Carga já Ocupada com Obrigatórias (Horas)
NOTA_CORTE_IA = 25       # Rigor da IA (Nota de Corte % de Similaridade)

# 4. Modo do Sistema
TRAVA_RIGIDA = False     # False = Exploratório (Avisa o gap), True = Matrícula Real (Bloqueia)


# ==============================================================================
# EXECUÇÃO DO MOTOR DE OTIMIZAÇÃO (O Problema da Mochila)
# ==============================================================================
from sentence_transformers import util

# Processamento das entradas
historico_aluno = [c.strip().upper() for c in HISTORICO.split(',') if c.strip()]
objetivo_texto = descricoes_areas.get(TRILHA_DESEJADA, "Engenharia de Produção")
vetor_foco = modelo_ia.encode(objetivo_texto)

capacidade_livre = CARGA_MAXIMA - CARGA_OBRIGATORIAS

print("\n" + "="*80)
print(" ANÁLISE DO PROBLEMA DA MOCHILA (SAD)")
print("="*80)
print(f"Espaço Livre para Optativas: {capacidade_livre}h")
print(f"Modo da IA: Aceitando matérias com Match >= {NOTA_CORTE_IA}%")
print(f"Trava de Requisitos: {'LIGADA (Restrito)' if TRAVA_RIGIDA else 'DESLIGADA (Exploratório)'}")

if capacidade_livre <= 0:
    print("\n⚠️ ALERTA: Não há espaço na carga horária para optativas.")
else:
    possiveis = []
    bloqueadas_interesse = []

    # Iteração sobre o dicionário de disciplinas (ETL prévio)
    for cod, dados in dict_disciplinas.items():
        if cod not in historico_aluno:
            vetor_ementa = dados.get('Vetor_Ementa')

            if vetor_ementa is not None:
                sim = util.cos_sim(vetor_foco, vetor_ementa).item()
                nota_match = max(0, sim * 100)

                if nota_match >= NOTA_CORTE_IA:
                    requisitos_disciplina = dados.get(col_req, '')

                    # Verifica a matemática dos requisitos
                    tem_requisito = checar_requisitos(requisitos_disciplina, historico_aluno)

                    # Lógica de Trava ou Exploratório
                    if TRAVA_RIGIDA and not tem_requisito:
                        bloqueadas_interesse.append({
                            'Codigo': cod, 'Nome': dados.get(col_nome, 'Desconhecido'),
                            'Requisitos': requisitos_disciplina, 'Score': nota_match
                        })
                    else:
                        possiveis.append({
                            'Codigo': cod, 'Nome': dados.get(col_nome, 'Desconhecido'),
                            'Horas': dados.get(col_cred, 60), 'Score': nota_match,
                            'Alerta': not tem_requisito
                        })

    # Heurística Gulosa: Ordena pelo maior Score
    possiveis = sorted(possiveis, key=lambda x: x['Score'], reverse=True)
    horas_acumuladas = 0

    print(f"\n GRADE OTIMIZADA PARA: {TRILHA_DESEJADA}\n")

    if not possiveis:
         print(" O sistema não encontrou disciplinas com essa nota de corte semântica.")
    else:
        for p in possiveis:
            # Tenta preencher a "Mochila"
            if (horas_acumuladas + p['Horas']) <= capacidade_livre:
                nota = round(p['Score'], 1)

                if nota >= 65: tag = "🎯 Foco Principal"
                elif nota >= 50: tag = "🟢 Forte Afinidade"
                else: tag = "🟡 Matéria Correlata"

                aviso_req = " [⚠️ FALTAM PRÉ-REQUISITOS]" if p['Alerta'] else ""

                print(f" {p['Codigo']} - {p['Nome']} ({p['Horas']}h) | Match: {nota}% -> {tag}{aviso_req}")
                horas_acumuladas += p['Horas']

    print("-" * 80)
    print(f"📌 Total Alocado: {horas_acumuladas}h do seu tempo livre.")

    # Visão de Futuro (Gap Analysis)
    if TRAVA_RIGIDA and bloqueadas_interesse:
        print("\n" + "="*80)
        print("🔮 GAP ANALYSIS: MATÉRIAS IDEAIS BLOQUEADAS")
        print("="*80)

        bloqueadas_interesse = sorted(bloqueadas_interesse, key=lambda x: x['Score'], reverse=True)[:4]
        for b in bloqueadas_interesse:
            nota_b = round(b['Score'], 1)
            req_b = b['Requisitos'] if b['Requisitos'] else "Não especificado"
            print(f"🚫 {b['Codigo']} - {b['Nome']} (Match: {nota_b}%) -> Exige: {req_b}")


 ANÁLISE DO PROBLEMA DA MOCHILA (SAD)
Espaço Livre para Optativas: 120h
Modo da IA: Aceitando matérias com Match >= 25%
Trava de Requisitos: DESLIGADA (Exploratório)

 GRADE OTIMIZADA PARA: Logística

 ENM0080 - FENOMENOS DE TRANSPORTE (75h) | Match: 66.5% -> 🎯 Foco Principal [⚠️ FALTAM PRÉ-REQUISITOS]
 EPR0040 - ERGONOMIA EM SISTEMAS PRODUTIVOS (30h) | Match: 47.3% -> 🟡 Matéria Correlata [⚠️ FALTAM PRÉ-REQUISITOS]
 DEG0200 - ATIVIDADE COMPLEMENTAR (15h) | Match: 29.5% -> 🟡 Matéria Correlata
--------------------------------------------------------------------------------
📌 Total Alocado: 120h do seu tempo livre.


In [5]:
# @title
# Conta quantas matérias têm texto na ementa para a IA ler
vetores_validos = sum(1 for d in dict_disciplinas.values() if d.get('Vetor_Ementa') is not None)
print(f"Total de disciplinas lidas do CSV: {len(dict_disciplinas)}")
print(f"Total de disciplinas com EMENTA VÁLIDA para a IA: {vetores_validos}")

Total de disciplinas lidas do CSV: 223
Total de disciplinas com EMENTA VÁLIDA para a IA: 223
